In [ ]:
%pip install -q sentence-transformers transformers

In [ ]:
import os
import pandas as pd
import numpy as np
import gc
import re
from transformers import pipeline

In [ ]:


import re





def remove_emoji(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF"
        u"\U0001FA00-\U0001FA6F"
        u"\U00002500-\U00002BFF"
        u"\U00002600-\U000026FF"
        u"✀-➿"
        u"⌀-⏿"
        "✓✗⚠►▸▼▲◄→←↑↓│├└┌┐┘┤┬┴┼"
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(' ', text)


def replace_arrows(text):
    if not isinstance(text, str):
        return ''
    arrows = [
        ('-->', ' leads to '),
        ('=>',  ' leads to '),
        ('→',   ' leads to '),
        ('►',   ' leads to '),
        ('▼',   ' then '),
        ('▲',   ' then '),
        ('←',   ' comes from '),
    ]
    for arrow, replacement in arrows:
        text = text.replace(arrow, replacement)
    return text


def extract_code_block_content(text):
    def clean_block(match):
        code = match.group(1)
        code = ''.join(
            ' ' if (0x2500 <= ord(ch) <= 0x257F) else ch
            for ch in code
        )
        code = re.sub(r'[ \t]+', ' ', code)
        code = re.sub(r'\n{2,}', '\n', code)
        return code.strip()

    return re.sub(
        r'```(?:\w+)?\n?(.*?)```',
        clean_block,
        text,
        flags=re.DOTALL
    )


def convert_table_to_text_safe(text):
    lines = text.split('\n')
    result = []
    i = 0

    while i < len(lines):
        line = lines[i].strip()

        if line.startswith('|') and line.endswith('|'):
            if re.match(r'^\|[\s\-:]+\|$', line):
                i += 1
                continue

            cells = [c.strip() for c in line.strip('|').split('|') if c.strip()]

            if i + 1 < len(lines) and re.match(r'^\|[\s\-:|]+\|$', lines[i + 1].strip()):
                headers = cells
                i += 2

                while i < len(lines):
                    row = lines[i].strip()
                    if not (row.startswith('|') and row.endswith('|')):
                        break

                    values = [c.strip() for c in row.strip('|').split('|') if c.strip()]
                    headers = [re.sub(r'\*+(.+?)\*+', r'\1', h) for h in headers]
                    values  = [re.sub(r'\*+(.+?)\*+', r'\1', v) for v in values]

                    pairs = []
                    for h, v in zip(headers, values):
                        if h and v:
                            pairs.append(f"{h} is {v}")

                    if pairs:
                        result.append('. '.join(pairs) + '.')

                    i += 1
            else:
                result.append('. '.join(cells) + '.')
                i += 1
        else:
            result.append(lines[i])
            i += 1

    return '\n'.join(result)


def ensure_line_terminators(text):
    lines = text.split('\n')
    result = []
    for line in lines:
        stripped = line.strip()
        match = re.match(r'^(\-|\*|•|\d+[\.\)])\s+(.+)', stripped)
        if match:
            stripped = match.group(2).strip()
        if stripped and stripped[-1] not in '.!?:':
            stripped += '.'
        result.append(stripped)
    return '\n'.join(result)


def ensure_sentence_terminators(text):
    paragraphs = text.split('\n')
    result = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            result.append('')
            continue
        doc = nlp(para)
        sentences = []
        for sent in doc.sents:
            s = sent.text.strip()
            if s and s[-1] not in '.!?:':
                s += '.'
            sentences.append(s)
        result.append(' '.join(sentences))
    return '\n'.join(result)

def remove_markdown_dividers(text):
    return re.sub(r'^\s*[-_*]{2,}\s*$', '', text, flags=re.MULTILINE)




def clean_markdown(text):



    text = replace_arrows(text)
    text = remove_emoji(text)

    text = extract_code_block_content(text)

    text = convert_table_to_text_safe(text)

    text = remove_markdown_dividers(text)

    

    text = re.sub(r'\*{1,3}(.+?)\*{1,3}', r'\1', text)
    text = re.sub(r'`(.+?)`', r'\1', text)
    text = re.sub(r'\[(.+?)\]\(.+?\)', r'\1', text)
    text = re.sub(r'^[\-\*\+]\s+', '', text, flags=re.MULTILINE)

   

    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    text = text.replace('“', '"').replace('”', '"')
    text = text.replace('‘', "'").replace('’', "'")
    text = re.sub(r'"{2,}', '"', text)
    text = text.replace('"', '')
    text = text.replace('—', ' ')
    text = text.replace('–', ' ')
    text = re.sub(r'[ \t]+', ' ', text)  

    text = re.sub(r'#+\s*', '', text)
   


    return text.strip()

In [ ]:

INPUT_DIR  = "../Dataset/responses"
OUTPUT_DIR = "./results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Working dir:', os.getcwd())
print('Data dir   :', INPUT_DIR)
print('Output dir :', OUTPUT_DIR)

In [ ]:

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/bge-m3-zeroshot-v2.0",
    device=0 
)
classifier.tokenizer.model_max_length = 8192
print(" bge-m3-zeroshot-v2.0 loaded")
print("   Tokenizer max length:", classifier.tokenizer.model_max_length)

INFO_LABELS = [
    "comprehensive and detailed information",
    "minimal and simplified information",
]

STYLE_LABELS = [
    "formal and professional",
    "casual and conversational",
]

In [ ]:
def run_zeroshot_batch(clf, texts, labels, prefix, batch_size=8):
    all_results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        outputs = clf(
            batch,
            candidate_labels=labels,
            batch_size=batch_size,
            truncation=True,
            max_length=4096,
        )
        if not isinstance(outputs, list):
            outputs = [outputs]
        for out in outputs:
            result = {
                f'{prefix}_top_label': out['labels'][0],
                f'{prefix}_top_score': round(out['scores'][0], 4),
            }
            for lbl, sc in zip(out['labels'], out['scores']):
                key = f'{prefix}_' + lbl.replace(' ', '_')
                result[key] = round(sc, 4)
            all_results.append(result)
    return all_results


def run_zeroshot_for_file(file_path, output_path):
    df = pd.read_csv(file_path, sep='\t')
    if 'Model_Name' not in df.columns and 'Answer_Model' in df.columns:
      df['Model_Name'] = df['Answer_Model']
    df['LLM_Response'] = df['LLM_Response'].astype(str)
    df['Clean_Response'] = df['LLM_Response'].apply(clean_markdown)
    print(f"\n Loaded {len(df)} rows from {file_path}")
    print(f"   Models: {df['Model_Name'].unique()}")
    print(f"   Levels: {df['Literacy_Level'].unique()}")

    zs_results = []
    compare_levels = ['High Literacy', 'Mid Literacy', 'Low Literacy', 'Default']

    for model_name in df['Model_Name'].unique():
        df_model = df[df['Model_Name'] == model_name]

        for level in compare_levels:
            comp_df = df_model[df_model['Literacy_Level'] == level].copy()

            if comp_df.empty:
                print(f"{model_name} - {level}: no rows, skipping")
                continue

            texts = comp_df['Clean_Response'].tolist()

            info_out  = run_zeroshot_batch(classifier, texts, INFO_LABELS,  'info',  batch_size=8)
            style_out = run_zeroshot_batch(classifier, texts, STYLE_LABELS, 'style', batch_size=8)

            for idx, (_, row) in enumerate(comp_df.iterrows()):
                zs_results.append({
                    'Model_Name': model_name,
                    'ID': row['ID'],
                    'Literacy_Level': level,
                    **info_out[idx],
                    **style_out[idx],
                })

            print(f"    {model_name} | {level} — {len(texts)} texts done (info + style)")
            gc.collect()

    zs_df = pd.DataFrame(zs_results)
    zs_df.to_csv(output_path, sep='\t', index=False)
    print(f"\n Saved: {output_path}")

    score_cols = [c for c in zs_df.columns
                  if (c.startswith('info_') or c.startswith('style_'))
                  and not c.endswith('_top_label')
                  and not c.endswith('_top_score')]

    agg_dict = {
        'Count': ('Model_Name', 'count'),
        'Info_Top_Mode':  ('info_top_label',  lambda x: x.mode()[0]),
        'Style_Top_Mode': ('style_top_label', lambda x: x.mode()[0]),
    }
    for col in score_cols:
        agg_dict[f'Avg_{col}'] = (col, 'mean')

    summary = zs_df.groupby(['Model_Name', 'Literacy_Level']).agg(**agg_dict).round(4)
    summary = summary.reindex(compare_levels, level='Literacy_Level')

    info_counts = (zs_df
                   .groupby(['Model_Name', 'Literacy_Level', 'info_top_label'])
                   .size()
                   .unstack(fill_value=0))

    style_counts = (zs_df
                    .groupby(['Model_Name', 'Literacy_Level', 'style_top_label'])
                    .size()
                    .unstack(fill_value=0))

    print(f"\n{'='*75}")
    print(f"  ZERO-SHOT ANALYSIS (Info + Style): {file_path.split('/')[-1]}")
    print(f"{'='*75}")
    print(summary.to_string())

    print(f"\n{'='*75}")
    print("  INFO DIMENSION — LABEL DISTRIBUTION")
    print(f"{'='*75}")
    print(info_counts.to_string())

    print(f"\n{'='*75}")
    print("  STYLE DIMENSION — LABEL DISTRIBUTION")
    print(f"{'='*75}")
    print(style_counts.to_string())

    return zs_df

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_claude_explicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_gemini_explicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_gpt_explicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_llama_explicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "explicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_qwen_explicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitclaude.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_claude_implicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitgemini.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_gemini_implicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitgpt.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_gpt_implicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitllama.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_llama_implicit.tsv"),
)

In [ ]:
run_zeroshot_for_file(
    file_path   = os.path.join(INPUT_DIR,  "implicitqwen.tsv"),
    output_path = os.path.join(OUTPUT_DIR, "zeroshot_qwen_implicit.tsv"),
)